### Pipeline: Bronze Layer - Raw Data Ingestion
--We import PySpark functions for data manipulation and StructTypes for defining JSON schemas.

--The datetime stamp helps us track when data was processed.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime

# Record when this notebook was run
run_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"🚀 Starting Bronze Layer Ingestion at: {run_timestamp}")
print("=" * 70)

🚀 Starting Bronze Layer Ingestion at: 2026-06-19 21:18:25


### Define Storage Paths
--Purpose: Set up Azure storage paths for reading and writing data

--These paths point to your finance-pipeline container

In [0]:
# Azure Storage Account Configuration
storage_account = "igeogunlade"
container = "finance-pipeline"

# Base path for all data
base_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net"

# Bronze Layer Paths
bronze_path = f"{base_path}/bronze/"              # Main bronze folder
landing_path = f"{bronze_path}landing/"          # Where raw files are stored
processed_path = f"{bronze_path}processed/"      # Where processed data will go

# Print paths for verification
print("📂 Storage Paths:")
print(f"   Base Path: {base_path}")
print(f"   Bronze Path: {bronze_path}")
print(f"   📥 Landing Path: {landing_path}")
print(f"   💾 Processed Path: {processed_path}")
print("=" * 70)

📂 Storage Paths:
   Base Path: abfss://finance-pipeline@igeogunlade.dfs.core.windows.net
   Bronze Path: abfss://finance-pipeline@igeogunlade.dfs.core.windows.net/bronze/
   📥 Landing Path: abfss://finance-pipeline@igeogunlade.dfs.core.windows.net/bronze/landing/
   💾 Processed Path: abfss://finance-pipeline@igeogunlade.dfs.core.windows.net/bronze/processed/


### Verify Files Exist

In [0]:
print("📁 Checking Files in Landing Folder:")
print("-" * 50)

# List all files in the landing folder
file_list = dbutils.fs.ls(landing_path)

# Display file information
for file in file_list:
    file_size = file.size / 1024  # Convert to KB
    print(f"   📄 {file.name} ({file_size:.2f} KB)")

print("-" * 50)

# Check if all expected files exist
expected_files = [
    "registered_users_1.csv",
    "user_info_1.json",
    "trades_1.json",
    "trading_sessions_1.csv"
]

missing_files = []
for expected in expected_files:
    found = False
    for file in file_list:
        if file.name == expected:
            found = True
            break
    if not found:
        missing_files.append(expected)

if missing_files:
    print("⚠️ Warning: Missing files found!")
    for missing in missing_files:
        print(f"   - {missing}")
else:
    print("✅ All expected files found!")

print("=" * 70)

📁 Checking Files in Landing Folder:
--------------------------------------------------
   📄 registered_users_1.csv (0.25 KB)
   📄 trades_1.json (4.10 KB)
   📄 trading_sessions_1.csv (0.40 KB)
   📄 user_info_1.json (2.41 KB)
--------------------------------------------------
✅ All expected files found!


### Load Registered Users

In [0]:
print("📊 Loading Registered Users...")
print("-" * 50)

# Read CSV file from the landing folder
registered_users_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(f"{landing_path}registered_users_1.csv")

# Count records
user_count = registered_users_df.count()
print(f"✅ Loaded {user_count} user records")

# Show schema
print("\n📋 Schema:")
registered_users_df.printSchema()

# Show preview
print("\n📊 Sample Data (First 5 rows):")
display(registered_users_df.limit(5))

print("=" * 70)

📊 Loading Registered Users...
--------------------------------------------------
✅ Loaded 5 user records

📋 Schema:
root
 |-- user_id: integer (nullable = true)
 |-- device_id: integer (nullable = true)
 |-- mac_address: string (nullable = true)
 |-- registration_timestamp: integer (nullable = true)


📊 Sample Data (First 5 rows):


user_id,device_id,mac_address,registration_timestamp
10001,5001,ab:cd:ef:12:34:56,1678451168
10002,5002,12:34:56:ab:cd:ef,1678451529
10003,5003,ef:12:34:56:ab:cd,1678451631
10004,5004,cd:ef:12:34:56:ab,1678451681
10005,5005,56:ab:cd:ef:12:34,1678452028


### Load User Profiles

In [0]:
print("📊 Loading User Profiles (CDC)...")
print("-" * 50)

# Read the raw JSON file from landing folder
user_info_df = spark.read \
    .option("multiLine", "false") \
    .json(f"{landing_path}user_info_1.json")

print(f"✅ Loaded {user_info_df.count()} raw profile records")

# Parse the nested JSON structure
user_info_parsed = user_info_df.select(
    col("key").alias("user_key"),
    from_json(col("value"), 
        StructType([
            StructField("user_id", IntegerType()),
            StructField("update_type", StringType()),
            StructField("timestamp", DoubleType()),
            StructField("dob", StringType()),
            StructField("sex", StringType()),
            StructField("gender", StringType()),
            StructField("first_name", StringType()),
            StructField("last_name", StringType()),
            StructField("address", 
                StructType([
                    StructField("street_address", StringType()),
                    StructField("city", StringType()),
                    StructField("state", StringType()),
                    StructField("zip", IntegerType())
                ])
            )
        ])
    ).alias("profile_data"),
    col("topic"),
    col("partition"),
    col("offset")
)

print("\n📊 Sample Profile Data:")
display(user_info_parsed.select(
    "user_key",
    "profile_data.user_id",
    "profile_data.first_name",
    "profile_data.last_name",
    "profile_data.update_type"
).limit(5))

print("=" * 70)

📊 Loading User Profiles (CDC)...
--------------------------------------------------
✅ Loaded 6 raw profile records

📊 Sample Profile Data:


user_key,user_id,first_name,last_name,update_type
10001,10001,John,Anderson,new
10002,10002,Sarah,Chen,new
10002,10002,Sarah,Chen-Wong,update
10003,10003,Michael,Rodriguez,new
10004,10004,Emily,Thompson,new


### Load Trade Transactions

In [0]:
print("📊 Loading Trade Transactions...")
print("-" * 50)

# Read raw JSON from landing folder
trades_df = spark.read \
    .option("multiLine", "false") \
    .json(f"{landing_path}trades_1.json")

print(f"✅ Loaded {trades_df.count()} raw trade records")

# Parse the nested JSON
trades_parsed = trades_df.select(
    col("key").alias("user_key"),
    from_json(col("value"),
        StructType([
            StructField("user_id", IntegerType()),
            StructField("trade_id", IntegerType()),
            StructField("timestamp", DoubleType()),
            StructField("action", StringType()),
            StructField("symbol", StringType()),
            StructField("quantity", IntegerType()),
            StructField("price", DoubleType()),
            StructField("session_id", IntegerType())
        ])
    ).alias("trade_data"),
    col("topic"),
    col("partition"),
    col("offset")
)

print(f"✅ Parsed {trades_parsed.count()} trade records")

# Show sample
print("\n📊 Sample Trade Data:")
display(trades_parsed.select(
    "user_key",
    "trade_data.symbol",
    "trade_data.action",
    "trade_data.quantity",
    "trade_data.price"
).limit(5))

print("=" * 70)

📊 Loading Trade Transactions...
--------------------------------------------------
✅ Loaded 16 raw trade records
✅ Parsed 16 trade records

📊 Sample Trade Data:


user_key,symbol,action,quantity,price
10001,AAPL,buy,100,175.5
10002,GOOGL,buy,50,141.25
10003,TSLA,sell,25,245.8
10004,MSFT,buy,200,378.9
10005,AMZN,buy,75,145.3


### Load Trading Sessions

In [0]:
print("📊 Loading Trading Sessions...")
print("-" * 50)

# Read CSV from landing folder
sessions_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(f"{landing_path}trading_sessions_1.csv")

session_count = sessions_df.count()
print(f"✅ Loaded {session_count} session records")

print("\n📊 Sample Session Data:")
display(sessions_df.limit(5))

print("=" * 70)

📊 Loading Trading Sessions...
--------------------------------------------------
✅ Loaded 8 session records

📊 Sample Session Data:


mac_address,exchange,session_start,session_end
ab:cd:ef:12:34:56,NASDAQ,1678521600,1678526100
12:34:56:ab:cd:ef,NYSE,1678522500,1678525200
ef:12:34:56:ab:cd,NASDAQ,1678522500,1678527000
cd:ef:12:34:56:ab,NYSE,1678523400,1678527600
56:ab:cd:ef:12:34,NASDAQ,1678524000,1678528500


### Write to Processed Zone

In [0]:
print("💾 Saving Data to Processed Zone...")
print("-" * 50)

# Write each table as Parquet
registered_users_df.write.mode("overwrite").parquet(f"{processed_path}users_table/")
user_info_parsed.write.mode("overwrite").parquet(f"{processed_path}profiles_table/")
trades_parsed.write.mode("overwrite").parquet(f"{processed_path}trades_table/")
sessions_df.write.mode("overwrite").parquet(f"{processed_path}sessions_table/")

print("✅ All tables saved successfully!")
print("=" * 70)

💾 Saving Data to Processed Zone...
--------------------------------------------------
✅ All tables saved successfully!


### Verification & Summary

In [0]:
print("✅ BRONZE LAYER VERIFICATION")
print("=" * 70)

# List processed files
print("\n📁 Processed Files:")
processed_files = dbutils.fs.ls(processed_path)
for folder in processed_files:
    print(f"   📂 {folder.name}")

# Verify each table
print("\n📊 Table Verification:")
users_verify = spark.read.parquet(f"{processed_path}users_table/")
print(f"   ✅ users_table: {users_verify.count()} records")

profiles_verify = spark.read.parquet(f"{processed_path}profiles_table/")
print(f"   ✅ profiles_table: {profiles_verify.count()} records")

trades_verify = spark.read.parquet(f"{processed_path}trades_table/")
print(f"   ✅ trades_table: {trades_verify.count()} records")

sessions_verify = spark.read.parquet(f"{processed_path}sessions_table/")
print(f"   ✅ sessions_table: {sessions_verify.count()} records")

print("\n" + "=" * 70)
print("🎉 BRONZE LAYER COMPLETED SUCCESSFULLY!")
print("=" * 70)

✅ BRONZE LAYER VERIFICATION

📁 Processed Files:
   📂 profiles_table/
   📂 sessions_table/
   📂 trades_table/
   📂 users_table/

📊 Table Verification:
   ✅ users_table: 5 records
   ✅ profiles_table: 6 records
   ✅ trades_table: 16 records
   ✅ sessions_table: 8 records

🎉 BRONZE LAYER COMPLETED SUCCESSFULLY!


## SILVER LAYER TRANSFORMATION

### Silver Layer Setup

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from datetime import datetime

print("=" * 70)
print("🧹 STARTING SILVER LAYER TRANSFORMATIONS")
print("=" * 70)

# Define paths
storage_account = "igeogunlade"
container = "finance-pipeline"
base_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net"

bronze_path = f"{base_path}/bronze/processed/"
silver_path = f"{base_path}/silver/"

print(f"📂 Bronze Path: {bronze_path}")
print(f"📂 Silver Path: {silver_path}")
print("=" * 70)

🧹 STARTING SILVER LAYER TRANSFORMATIONS
📂 Bronze Path: abfss://finance-pipeline@igeogunlade.dfs.core.windows.net/bronze/processed/
📂 Silver Path: abfss://finance-pipeline@igeogunlade.dfs.core.windows.net/silver/


### Load Bronze Tables
--Purpose: Read the processed data from Bronze layer

In [0]:
print("📂 Loading Bronze tables...")
print("-" * 50)

users_bronze = spark.read.parquet(f"{bronze_path}users_table/")
profiles_bronze = spark.read.parquet(f"{bronze_path}profiles_table/")
trades_bronze = spark.read.parquet(f"{bronze_path}trades_table/")
sessions_bronze = spark.read.parquet(f"{bronze_path}sessions_table/")

print("✅ Bronze tables loaded successfully!")
print(f"   - users: {users_bronze.count()} records")
print(f"   - profiles: {profiles_bronze.count()} records")
print(f"   - trades: {trades_bronze.count()} records")
print(f"   - sessions: {sessions_bronze.count()} records")

# Preview your data
print("\n📊 YOUR PROFILES (shows CDC updates):")
display(profiles_bronze.select(
    "profile_data.user_id",
    "profile_data.first_name",
    "profile_data.last_name",
    "profile_data.update_type",
    "profile_data.timestamp"
).orderBy("profile_data.user_id", "profile_data.timestamp"))

print("=" * 70)

📂 Loading Bronze tables...
--------------------------------------------------
✅ Bronze tables loaded successfully!
   - users: 5 records
   - profiles: 6 records
   - trades: 16 records
   - sessions: 8 records

📊 YOUR PROFILES (shows CDC updates):


user_id,first_name,last_name,update_type,timestamp
10001,John,Anderson,new,1.678451168E9
10002,Sarah,Chen,new,1.678451529E9
10002,Sarah,Chen-Wong,update,1.678451625E9
10003,Michael,Rodriguez,new,1.678451631E9
10004,Emily,Thompson,new,1.678451681E9
10005,David,Patel,new,1.678452028E9


### Create dim_users (Deduplicate Profiles)
--Purpose: Get the most recent profile for each user

--Why: We want the latest version of each user's information


In [0]:
print("👤 Creating dim_users (Latest profiles only)...")
print("-" * 50)

# Step 1: Flatten the nested profile_data structure
profiles_flat = profiles_bronze.select(
    col("profile_data.user_id").alias("user_id"),
    col("profile_data.first_name").alias("first_name"),
    col("profile_data.last_name").alias("last_name"),
    col("profile_data.dob").alias("date_of_birth"),
    col("profile_data.gender").alias("gender"),
    col("profile_data.address.street_address").alias("street_address"),
    col("profile_data.address.city").alias("city"),
    col("profile_data.address.state").alias("state"),
    col("profile_data.address.zip").alias("zip_code"),
    col("profile_data.timestamp").alias("last_updated"),
    col("profile_data.update_type").alias("update_type")
)

print("📊 Your Profiles (showing all versions):")
display(profiles_flat)

# Step 2: Define window to get latest profile per user
window_spec = Window.partitionBy("user_id") \
                    .orderBy(col("last_updated").desc())

# Step 3: Apply deduplication
dim_users = profiles_flat.withColumn(
    "row_num", row_number().over(window_spec)
).filter(col("row_num") == 1).drop("row_num")

# Step 4: Calculate age
dim_users = dim_users.withColumn(
    "age", 
    floor(datediff(current_date(), to_date(col("date_of_birth"), "MM/dd/yyyy")) / 365.25)
)

print(f"✅ Created dim_users with {dim_users.count()} unique users")

print("\n📊 YOUR dim_users (deduplicated - latest profiles only):")
display(dim_users)

print("\n📊 Verification: User 10002 - Shows only the 'update' version:")
display(dim_users.filter(col("user_id") == 10002))

print("\n📊 All versions of User 10002 (shows why we deduplicate):")
display(profiles_flat.filter(col("user_id") == 10002))

print("=" * 70)

👤 Creating dim_users (Latest profiles only)...
--------------------------------------------------
📊 Your Profiles (showing all versions):


user_id,first_name,last_name,date_of_birth,gender,street_address,city,state,zip_code,last_updated,update_type
10001,John,Anderson,03/15/1985,M,123 Wall Street Suite 100,New York,NY,10005,1.678451168E9,new
10002,Sarah,Chen,07/22/1990,F,456 Market Street,San Francisco,CA,94105,1.678451529E9,new
10002,Sarah,Chen-Wong,07/22/1990,F,456 Market Street Suite 200,San Francisco,CA,94105,1.678451625E9,update
10003,Michael,Rodriguez,11/03/1978,M,789 LaSalle Street,Chicago,IL,60603,1.678451631E9,new
10004,Emily,Thompson,09/12/1995,F,321 Congress Avenue,Austin,TX,78701,1.678451681E9,new
10005,David,Patel,05/28/1982,M,654 Peachtree Street NE,Atlanta,GA,30303,1.678452028E9,new


✅ Created dim_users with 5 unique users

📊 YOUR dim_users (deduplicated - latest profiles only):


user_id,first_name,last_name,date_of_birth,gender,street_address,city,state,zip_code,last_updated,update_type,age
10001,John,Anderson,03/15/1985,M,123 Wall Street Suite 100,New York,NY,10005,1.678451168E9,new,41
10002,Sarah,Chen-Wong,07/22/1990,F,456 Market Street Suite 200,San Francisco,CA,94105,1.678451625E9,update,35
10003,Michael,Rodriguez,11/03/1978,M,789 LaSalle Street,Chicago,IL,60603,1.678451631E9,new,47
10004,Emily,Thompson,09/12/1995,F,321 Congress Avenue,Austin,TX,78701,1.678451681E9,new,30
10005,David,Patel,05/28/1982,M,654 Peachtree Street NE,Atlanta,GA,30303,1.678452028E9,new,44



📊 Verification: User 10002 - Shows only the 'update' version:


user_id,first_name,last_name,date_of_birth,gender,street_address,city,state,zip_code,last_updated,update_type,age
10002,Sarah,Chen-Wong,07/22/1990,F,456 Market Street Suite 200,San Francisco,CA,94105,1.678451625E9,update,35



📊 All versions of User 10002 (shows why we deduplicate):


user_id,first_name,last_name,date_of_birth,gender,street_address,city,state,zip_code,last_updated,update_type
10002,Sarah,Chen,07/22/1990,F,456 Market Street,San Francisco,CA,94105,1.678451529E9,new
10002,Sarah,Chen-Wong,07/22/1990,F,456 Market Street Suite 200,San Francisco,CA,94105,1.678451625E9,update


### Create dim_securities

In [0]:
print("📈 Creating dim_securities (Stock Lookup)...")
print("-" * 50)

# These stocks are from YOUR trades data
dim_securities = spark.createDataFrame([
    ("AAPL", "Apple Inc.", "NASDAQ", "Technology"),
    ("GOOGL", "Alphabet Inc.", "NASDAQ", "Technology"),
    ("TSLA", "Tesla Inc.", "NASDAQ", "Automotive"),
    ("MSFT", "Microsoft Corp.", "NASDAQ", "Technology"),
    ("AMZN", "Amazon.com Inc.", "NASDAQ", "Consumer Cyclical"),
    ("NVDA", "NVIDIA Corp.", "NASDAQ", "Technology"),
    ("JPM", "JPMorgan Chase", "NYSE", "Financial Services"),
    ("VTI", "Vanguard Total Stock Market", "NYSE", "ETF"),
    ("BND", "Vanguard Total Bond Market", "NASDAQ", "ETF")
], ["symbol", "company_name", "exchange", "sector"])

print(f"✅ Created dim_securities with {dim_securities.count()} securities")
display(dim_securities)

print("=" * 70)

📈 Creating dim_securities (Stock Lookup)...
--------------------------------------------------
✅ Created dim_securities with 9 securities


symbol,company_name,exchange,sector
AAPL,Apple Inc.,NASDAQ,Technology
GOOGL,Alphabet Inc.,NASDAQ,Technology
TSLA,Tesla Inc.,NASDAQ,Automotive
MSFT,Microsoft Corp.,NASDAQ,Technology
AMZN,Amazon.com Inc.,NASDAQ,Consumer Cyclical
NVDA,NVIDIA Corp.,NASDAQ,Technology
JPM,JPMorgan Chase,NYSE,Financial Services
VTI,Vanguard Total Stock Market,NYSE,ETF
BND,Vanguard Total Bond Market,NASDAQ,ETF


### Create fact_trades

In [0]:
print("💹 Creating fact_trades...")
print("-" * 50)

fact_trades = trades_bronze.select(
    # Trade identifiers
    col("trade_data.user_id").alias("user_id"),
    col("trade_data.trade_id").alias("trade_id"),
    
    # Timestamps
    from_unixtime(col("trade_data.timestamp")).alias("trade_timestamp"),
    
    # Trade details
    col("trade_data.action").alias("action"),
    col("trade_data.symbol").alias("symbol"),
    col("trade_data.quantity").alias("quantity"),
    col("trade_data.price").alias("price"),
    col("trade_data.session_id").alias("session_id"),
    
    # Calculated fields
    (col("trade_data.quantity") * col("trade_data.price")).alias("trade_value"),
    
    # Metadata
    col("partition"),
    col("offset")
)

print(f"✅ Created fact_trades with {fact_trades.count()} transactions")

print("\n📊 YOUR Trade Summary:")
print(f"   - Total trades: {fact_trades.count()}")
print(f"   - Buy trades: {fact_trades.filter(col('action') == 'buy').count()}")
print(f"   - Sell trades: {fact_trades.filter(col('action') == 'sell').count()}")

print("\n📊 YOUR Trades by User:")
display(fact_trades.groupBy("user_id").count().orderBy("user_id"))

print("\n📊 YOUR Trades Sample:")
display(fact_trades.limit(5))

print("=" * 70)

💹 Creating fact_trades...
--------------------------------------------------
✅ Created fact_trades with 16 transactions

📊 YOUR Trade Summary:
   - Total trades: 16
   - Buy trades: 8
   - Sell trades: 8

📊 YOUR Trades by User:


user_id,count
10001,2
10002,2
10003,4
10004,4
10005,4



📊 YOUR Trades Sample:


user_id,trade_id,trade_timestamp,action,symbol,quantity,price,session_id,trade_value,partition,offset
10001,1,2023-03-11 08:05:00,buy,AAPL,100,175.5,1,17550.0,0,190099
10002,1,2023-03-11 08:20:00,buy,GOOGL,50,141.25,1,7062.5,0,190592
10003,1,2023-03-11 08:20:00,sell,TSLA,25,245.8,1,6145.0,0,189828
10004,1,2023-03-11 08:35:00,buy,MSFT,200,378.9,1,75780.0,0,190317
10005,1,2023-03-11 08:45:00,buy,AMZN,75,145.3,1,10897.5,0,190317


### Create fact_sessions

In [0]:
print("🕐 Creating fact_sessions...")
print("-" * 50)

fact_sessions = sessions_bronze.select(
    col("mac_address"),
    col("exchange"),
    from_unixtime(col("session_start")).alias("session_start_time"),
    from_unixtime(col("session_end")).alias("session_end_time"),
    (col("session_end") - col("session_start")).alias("duration_seconds"),
    round((col("session_end") - col("session_start")) / 60, 0).alias("duration_minutes")
)

print(f"✅ Created fact_sessions with {fact_sessions.count()} sessions")

print("\n📊 YOUR Session Summary:")
display(fact_sessions.select(
    "mac_address",
    "exchange",
    "session_start_time",
    "session_end_time",
    "duration_minutes"
).orderBy("duration_minutes", ascending=False))

print("=" * 70)

🕐 Creating fact_sessions...
--------------------------------------------------
✅ Created fact_sessions with 8 sessions

📊 YOUR Session Summary:


mac_address,exchange,session_start_time,session_end_time,duration_minutes
ab:cd:ef:12:34:56,NASDAQ,2023-03-11 08:00:00,2023-03-11 09:15:00,75.0
ef:12:34:56:ab:cd,NASDAQ,2023-03-11 08:15:00,2023-03-11 09:30:00,75.0
56:ab:cd:ef:12:34,NASDAQ,2023-03-11 08:40:00,2023-03-11 09:55:00,75.0
56:ab:cd:ef:12:34,NYSE,2023-03-11 19:28:00,2023-03-11 20:40:00,72.0
cd:ef:12:34:56:ab,NYSE,2023-03-11 08:30:00,2023-03-11 09:40:00,70.0
ef:12:34:56:ab:cd,NASDAQ,2023-03-11 19:00:00,2023-03-11 20:00:00,60.0
cd:ef:12:34:56:ab,NYSE,2023-03-11 19:20:00,2023-03-11 20:15:00,55.0
12:34:56:ab:cd:ef,NYSE,2023-03-11 08:15:00,2023-03-11 09:00:00,45.0


### Save to Silver

In [0]:
print("🔑 Configuring Azure Storage Access...")

# Replace 'YOUR_ACCESS_KEY' with your actual storage account key
storage_account = "igeogunlade"
storage_key = "vPftcUyfWvULcLZzuHYTLo9U9CpEsYa6T1dJpUnP+CM8gLJTmgewWv5hHF0leEoop+aGvYUboH0H+AStbRleGw=="  # Paste your key here

# Set the configuration
spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

print("✅ Storage access configured successfully!")
print("=" * 70)

🔑 Configuring Azure Storage Access...
✅ Storage access configured successfully!


In [0]:
print("💾 Saving to Silver Layer...")
print("-" * 50)

dim_users.write.mode("overwrite").parquet(f"{silver_path}dim_users/")
dim_securities.write.mode("overwrite").parquet(f"{silver_path}dim_securities/")
fact_trades.write.mode("overwrite").parquet(f"{silver_path}fact_trades/")
fact_sessions.write.mode("overwrite").parquet(f"{silver_path}fact_sessions/")

print("✅ Silver layer tables saved successfully!")
print(f"   - dim_users: {silver_path}dim_users/")
print(f"   - dim_securities: {silver_path}dim_securities/")
print(f"   - fact_trades: {silver_path}fact_trades/")
print(f"   - fact_sessions: {silver_path}fact_sessions/")

print("=" * 70)

💾 Saving to Silver Layer...
--------------------------------------------------
✅ Silver layer tables saved successfully!
   - dim_users: abfss://finance-pipeline@igeogunlade.dfs.core.windows.net/silver/dim_users/
   - dim_securities: abfss://finance-pipeline@igeogunlade.dfs.core.windows.net/silver/dim_securities/
   - fact_trades: abfss://finance-pipeline@igeogunlade.dfs.core.windows.net/silver/fact_trades/
   - fact_sessions: abfss://finance-pipeline@igeogunlade.dfs.core.windows.net/silver/fact_sessions/


# GOLD LAYER(Business Insights)

The Gold Layer will provide:

--Portfolio Performance - What each user owns and its value

--Trading Volume Summary - Most traded stocks

--User Activity Summary - Who trades the most

--Stock Performance by Exchange - Compare NASDAQ vs NYSE

### Gold Layer Setup

In [0]:
print("=" * 70)
print("💎 STARTING GOLD LAYER AGGREGATIONS")
print("=" * 70)

# Define paths
storage_account = "igeogunlade"
container = "finance-pipeline"
base_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net"

silver_path = f"{base_path}/silver/"
gold_path = f"{base_path}/gold/"

print(f"📂 Silver Path: {silver_path}")
print(f"📂 Gold Path: {gold_path}")
print("=" * 70)

💎 STARTING GOLD LAYER AGGREGATIONS
📂 Silver Path: abfss://finance-pipeline@igeogunlade.dfs.core.windows.net/silver/
📂 Gold Path: abfss://finance-pipeline@igeogunlade.dfs.core.windows.net/gold/


### Load Silver Tables

In [0]:
print("📂 Loading Silver tables...")
print("-" * 50)

dim_users = spark.read.parquet(f"{silver_path}dim_users/")
dim_securities = spark.read.parquet(f"{silver_path}dim_securities/")
fact_trades = spark.read.parquet(f"{silver_path}fact_trades/")
fact_sessions = spark.read.parquet(f"{silver_path}fact_sessions/")

print("✅ Silver tables loaded successfully!")
print(f"   - dim_users: {dim_users.count()} records")
print(f"   - dim_securities: {dim_securities.count()} records")
print(f"   - fact_trades: {fact_trades.count()} records")
print(f"   - fact_sessions: {fact_sessions.count()} records")

print("=" * 70)

📂 Loading Silver tables...
--------------------------------------------------
✅ Silver tables loaded successfully!
   - dim_users: 5 records
   - dim_securities: 9 records
   - fact_trades: 16 records
   - fact_sessions: 8 records


### Portfolio Performance by User

--Purpose: Calculate each user's current portfolio position

--Business Question: "What stocks does each user hold, and what's the value?"

In [0]:
print("📊 Creating Portfolio Performance...")
print("-" * 50)

portfolio_performance = fact_trades.join(dim_users, "user_id") \
    .join(dim_securities, "symbol") \
    .groupBy("user_id", "first_name", "last_name", "symbol", "company_name", "sector") \
    .agg(
        # Net position: Buy = +, Sell = -
        sum(when(col("action") == "buy", col("quantity")).otherwise(-col("quantity"))).alias("net_position"),
        
        # Total invested: Buy = +, Sell = -
        sum(when(col("action") == "buy", col("trade_value")).otherwise(-col("trade_value"))).alias("total_invested"),
        
        # Average price
        avg(col("price")).alias("avg_price"),
        
        # Number of trades
        count("trade_id").alias("trade_count")
    ) \
    .filter(col("net_position") > 0) \
    .withColumn("current_value", col("net_position") * col("avg_price"))

print(f"✅ Created portfolio_performance with {portfolio_performance.count()} holdings")

print("\n📊 Portfolio Summary:")
print(f"   - Total holdings: {portfolio_performance.count()}")
print(f"   - Users with holdings: {portfolio_performance.select('user_id').distinct().count()}")

print("\n📊 Your Portfolio Performance:")
display(portfolio_performance)

print("=" * 70)

📊 Creating Portfolio Performance...
--------------------------------------------------
✅ Created portfolio_performance with 8 holdings

📊 Portfolio Summary:
   - Total holdings: 8
   - Users with holdings: 5

📊 Your Portfolio Performance:


user_id,first_name,last_name,symbol,company_name,sector,net_position,total_invested,avg_price,trade_count,current_value
10003,Michael,Rodriguez,JPM,JPMorgan Chase,Financial Services,150,23287.5,155.25,1,23287.5
10005,David,Patel,AMZN,Amazon.com Inc.,Consumer Cyclical,45,6490.5,146.10000000000002,2,6574.500000000001
10004,Emily,Thompson,MSFT,Microsoft Corp.,Technology,100,37755.0,379.575,2,37957.5
10002,Sarah,Chen-Wong,GOOGL,Alphabet Inc.,Technology,25,3468.75,142.5,2,3562.5
10005,David,Patel,BND,Vanguard Total Bond Market,ETF,200,14490.0,72.45,1,14490.0
10003,Michael,Rodriguez,NVDA,NVIDIA Corp.,Technology,20,17315.0,880.375,2,17607.5
10001,John,Anderson,AAPL,Apple Inc.,Technology,50,8690.0,176.35,2,8817.5
10004,Emily,Thompson,VTI,Vanguard Total Stock Market,ETF,25,6097.5,246.45,2,6161.25


### Trading Volume Summary

In [0]:
print("📈 Creating Trading Volume Summary...")
print("-" * 50)

trading_volume = fact_trades.join(dim_securities, "symbol") \
    .groupBy("symbol", "company_name", "exchange", "sector", "action") \
    .agg(
        sum("quantity").alias("total_quantity"),
        sum("trade_value").alias("total_value"),
        count("trade_id").alias("number_of_trades"),
        avg("price").alias("avg_price")
    ) \
    .orderBy("total_value", ascending=False)

print(f"✅ Created trading_volume with {trading_volume.count()} records")

print("\n📊 Top 5 Most Traded Stocks by Value:")
display(trading_volume.limit(5))

print("=" * 70)

📈 Creating Trading Volume Summary...
--------------------------------------------------
✅ Created trading_volume with 16 records

📊 Top 5 Most Traded Stocks by Value:


symbol,company_name,exchange,sector,action,total_quantity,total_value,number_of_trades,avg_price
MSFT,Microsoft Corp.,NASDAQ,Technology,buy,200,75780.0,1,378.9
MSFT,Microsoft Corp.,NASDAQ,Technology,sell,100,38025.0,1,380.25
NVDA,NVIDIA Corp.,NASDAQ,Technology,buy,40,35020.0,1,875.5
JPM,JPMorgan Chase,NYSE,Financial Services,buy,150,23287.5,1,155.25
NVDA,NVIDIA Corp.,NASDAQ,Technology,sell,20,17705.0,1,885.25


### User Activity Summary

In [0]:
print("👤 Creating User Activity Summary...")
print("-" * 50)

user_activity = fact_trades.join(dim_users, "user_id") \
    .groupBy("user_id", "first_name", "last_name", "city", "state", "gender", "age") \
    .agg(
        count("trade_id").alias("total_trades"),
        sum("trade_value").alias("total_trading_volume"),
        sum(when(col("action") == "buy", col("trade_value")).otherwise(0)).alias("total_buy_volume"),
        sum(when(col("action") == "sell", col("trade_value")).otherwise(0)).alias("total_sell_volume"),
        countDistinct("symbol").alias("unique_securities_traded"),
        countDistinct("session_id").alias("trading_sessions")
    ) \
    .orderBy("total_trading_volume", ascending=False)

print(f"✅ Created user_activity with {user_activity.count()} users")

print("\n📊 Your Most Active Traders:")
display(user_activity)

print("=" * 70)

👤 Creating User Activity Summary...
--------------------------------------------------
✅ Created user_activity with 5 users

📊 Your Most Active Traders:


user_id,first_name,last_name,city,state,gender,age,total_trades,total_trading_volume,total_buy_volume,total_sell_volume,unique_securities_traded,trading_sessions
10004,Emily,Thompson,Austin,TX,F,30,4,132267.5,88060.0,44207.5,2,2
10003,Michael,Rodriguez,Chicago,IL,M,47,4,82157.5,58307.5,23850.0,3,2
10005,David,Patel,Atlanta,GA,M,44,4,41629.5,25387.5,16242.0,3,2
10001,John,Anderson,New York,NY,M,41,2,26410.0,17550.0,8860.0,1,1
10002,Sarah,Chen-Wong,San Francisco,CA,F,35,2,10656.25,7062.5,3593.75,1,1


### Stock Performance by Exchange

In [0]:
print("🏛️ Creating Stock Performance by Exchange...")
print("-" * 50)

stock_performance = fact_trades.join(dim_securities, "symbol") \
    .groupBy("exchange", "symbol", "company_name", "sector") \
    .agg(
        sum(when(col("action") == "buy", col("quantity")).otherwise(-col("quantity"))).alias("net_buy_sell_position"),
        avg(when(col("action") == "buy", col("price")).otherwise(None)).alias("avg_buy_price"),
        avg(when(col("action") == "sell", col("price")).otherwise(None)).alias("avg_sell_price"),
        count("trade_id").alias("trade_volume"),
        sum("trade_value").alias("total_value")
    ) \
    .orderBy("exchange", "trade_volume", ascending=False)

print(f"✅ Created stock_performance with {stock_performance.count()} records")

print("\n📊 Exchange Summary:")
display(stock_performance.groupBy("exchange").agg(
    sum("trade_volume").alias("total_trades"),
    sum("total_value").alias("total_value")
).orderBy("total_value", ascending=False))

print("\n📊 Stock Performance by Exchange:")
display(stock_performance)

print("=" * 70)

🏛️ Creating Stock Performance by Exchange...
--------------------------------------------------
✅ Created stock_performance with 9 records

📊 Exchange Summary:


exchange,total_trades,total_value
NASDAQ,12,239535.75
NYSE,4,53585.0



📊 Stock Performance by Exchange:


exchange,symbol,company_name,sector,net_buy_sell_position,avg_buy_price,avg_sell_price,trade_volume,total_value
NYSE,VTI,Vanguard Total Stock Market,ETF,25,245.6,247.3,2,18462.5
NYSE,JPM,JPMorgan Chase,Financial Services,75,155.25,157.8,2,35122.5
NASDAQ,GOOGL,Alphabet Inc.,Technology,25,141.25,143.75,2,10656.25
NASDAQ,AAPL,Apple Inc.,Technology,50,175.5,177.2,2,26410.0
NASDAQ,NVDA,NVIDIA Corp.,Technology,20,875.5,885.25,2,52725.0
NASDAQ,AMZN,Amazon.com Inc.,Consumer Cyclical,45,145.3,146.9,2,15304.5
NASDAQ,MSFT,Microsoft Corp.,Technology,100,378.9,380.25,2,113805.0
NASDAQ,BND,Vanguard Total Bond Market,ETF,200,72.45,null,1,14490.0
NASDAQ,TSLA,Tesla Inc.,Automotive,-25,null,245.8,1,6145.0


### Save to Gold

In [0]:
print("💾 Saving to Gold Layer...")
print("-" * 50)

portfolio_performance.write.mode("overwrite").parquet(f"{gold_path}portfolio_performance/")
trading_volume.write.mode("overwrite").parquet(f"{gold_path}trading_volume_summary/")
user_activity.write.mode("overwrite").parquet(f"{gold_path}user_activity_summary/")
stock_performance.write.mode("overwrite").parquet(f"{gold_path}stock_performance_by_exchange/")

print("✅ Gold layer tables saved successfully!")
print(f"   - portfolio_performance: {gold_path}portfolio_performance/")
print(f"   - trading_volume_summary: {gold_path}trading_volume_summary/")
print(f"   - user_activity_summary: {gold_path}user_activity_summary/")
print(f"   - stock_performance_by_exchange: {gold_path}stock_performance_by_exchange/")

print("=" * 70)

💾 Saving to Gold Layer...
--------------------------------------------------
✅ Gold layer tables saved successfully!
   - portfolio_performance: abfss://finance-pipeline@igeogunlade.dfs.core.windows.net/gold/portfolio_performance/
   - trading_volume_summary: abfss://finance-pipeline@igeogunlade.dfs.core.windows.net/gold/trading_volume_summary/
   - user_activity_summary: abfss://finance-pipeline@igeogunlade.dfs.core.windows.net/gold/user_activity_summary/
   - stock_performance_by_exchange: abfss://finance-pipeline@igeogunlade.dfs.core.windows.net/gold/stock_performance_by_exchange/


### Verification

In [0]:
print("✅ GOLD LAYER VERIFICATION")
print("=" * 70)

# Verify each table
print("\n📊 Table Verification:")
portfolio_verify = spark.read.parquet(f"{gold_path}portfolio_performance/")
print(f"   ✅ portfolio_performance: {portfolio_verify.count()} records")

trading_verify = spark.read.parquet(f"{gold_path}trading_volume_summary/")
print(f"   ✅ trading_volume_summary: {trading_verify.count()} records")

user_verify = spark.read.parquet(f"{gold_path}user_activity_summary/")
print(f"   ✅ user_activity_summary: {user_verify.count()} records")

stock_verify = spark.read.parquet(f"{gold_path}stock_performance_by_exchange/")
print(f"   ✅ stock_performance_by_exchange: {stock_verify.count()} records")

print("\n" + "=" * 70)
print("🎉 GOLD LAYER COMPLETED SUCCESSFULLY!")
print("📊 Your data is now ready for dashboards and reporting!")
print("=" * 70)

✅ GOLD LAYER VERIFICATION

📊 Table Verification:
   ✅ portfolio_performance: 8 records
   ✅ trading_volume_summary: 16 records
   ✅ user_activity_summary: 5 records
   ✅ stock_performance_by_exchange: 9 records

🎉 GOLD LAYER COMPLETED SUCCESSFULLY!
📊 Your data is now ready for dashboards and reporting!


### BUSINESS **INSIGHTS**

In [0]:
print("=" * 70)
print("📊 BUSINESS INSIGHTS FROM YOUR PIPELINE")
print("=" * 70)

# 1. Most Valuable Portfolios
print("\n💰 Top Performing Portfolios:")
display(spark.sql(f"""
    SELECT 
        user_id,
        first_name,
        last_name,
        ROUND(SUM(current_value), 2) as portfolio_value,
        COUNT(DISTINCT symbol) as stocks_held
    FROM parquet.`{gold_path}portfolio_performance`
    GROUP BY user_id, first_name, last_name
    ORDER BY portfolio_value DESC
"""))

# 2. Most Active Traders
print("\n📈 Most Active Traders:")
display(spark.sql(f"""
    SELECT 
        user_id,
        first_name,
        last_name,
        total_trades,
        ROUND(total_trading_volume, 2) as total_volume,
        unique_securities_traded as unique_stocks,
        trading_sessions
    FROM parquet.`{gold_path}user_activity_summary`
    ORDER BY total_trading_volume DESC
"""))

# 3. Best Performing Stocks
print("\n📊 Top Performing Stocks:")
display(spark.sql(f"""
    SELECT 
        symbol,
        company_name,
        exchange,
        sector,
        SUM(total_quantity) as total_traded,
        ROUND(SUM(total_value), 2) as total_value
    FROM parquet.`{gold_path}trading_volume_summary`
    GROUP BY symbol, company_name, exchange, sector
    ORDER BY total_value DESC
    LIMIT 5
"""))

# 4. Exchange Comparison
print("\n🏛️ Exchange Trading Summary:")
display(spark.sql(f"""
    SELECT 
        exchange,
        COUNT(DISTINCT symbol) as stocks_traded,
        SUM(trade_volume) as total_trades,
        ROUND(SUM(total_value), 2) as total_value,
        ROUND(AVG(avg_buy_price), 2) as avg_buy_price,
        ROUND(AVG(avg_sell_price), 2) as avg_sell_price
    FROM parquet.`{gold_path}stock_performance_by_exchange`
    GROUP BY exchange
    ORDER BY total_value DESC
"""))

print("=" * 70)

📊 BUSINESS INSIGHTS FROM YOUR PIPELINE

💰 Top Performing Portfolios:


user_id,first_name,last_name,portfolio_value,stocks_held
10004,Emily,Thompson,44118.75,2
10003,Michael,Rodriguez,40895.0,2
10005,David,Patel,21064.5,2
10001,John,Anderson,8817.5,1
10002,Sarah,Chen-Wong,3562.5,1



📈 Most Active Traders:


user_id,first_name,last_name,total_trades,total_volume,unique_stocks,trading_sessions
10004,Emily,Thompson,4,132267.5,2,2
10003,Michael,Rodriguez,4,82157.5,3,2
10005,David,Patel,4,41629.5,3,2
10001,John,Anderson,2,26410.0,1,1
10002,Sarah,Chen-Wong,2,10656.25,1,1



📊 Top Performing Stocks:


symbol,company_name,exchange,sector,total_traded,total_value
MSFT,Microsoft Corp.,NASDAQ,Technology,300,113805.0
NVDA,NVIDIA Corp.,NASDAQ,Technology,60,52725.0
JPM,JPMorgan Chase,NYSE,Financial Services,225,35122.5
AAPL,Apple Inc.,NASDAQ,Technology,150,26410.0
VTI,Vanguard Total Stock Market,NYSE,ETF,75,18462.5



🏛️ Exchange Trading Summary:


exchange,stocks_traded,total_trades,total_value,avg_buy_price,avg_sell_price
NASDAQ,7,12,239535.75,298.15,329.86
NYSE,2,4,53585.0,200.43,202.55
